In [1]:
"""
MỤC ĐÍCH:
  Nhận vào 3 file sạch từ Dataset trên Kaggle (/kaggle/input/...):
    1. Hanoi_Clean_Tabular.csv (Dữ liệu bảng)
    2. Hanoi_Clean_Text.csv    (Dữ liệu văn bản)
    3. Hanoi_Clean_Full.csv    (Dữ liệu đa phương thức)
  
  Tự động co giãn theo số lượng mẫu thực tế,
  phân chia Deduplicated Stratified 80/10/10 đồng bộ theo ID (random_state=42),
  và xuất ra ĐÚNG 9 FILE độc lập vào thư mục /kaggle/working/:
    * Nhóm Tabular (3 files) : Hanoi_Tabular_Train.csv, Hanoi_Tabular_Val.csv, Hanoi_Tabular_Test.csv
    * Nhóm Text (3 files)    : Hanoi_Text_Train.csv, Hanoi_Text_Val.csv, Hanoi_Text_Test.csv
    * Nhóm Full (3 files)    : Hanoi_Full_Train.csv, Hanoi_Full_Val.csv, Hanoi_Full_Test.csv
"""

import os
import glob
import csv
import random
from collections import defaultdict, Counter


def load_csv(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        return list(reader), reader.fieldnames


def write_csv(path, rows, fieldnames):
    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def find_input_directory():
    """
    Tự động tìm đường dẫn chứa 3 file input trên Kaggle hoặc máy local.
    Trên Kaggle: dữ liệu nằm trong /kaggle/input/<tên-dataset>/
    """
    # 1. Tìm kiếm đệ quy trong /kaggle/input/
    matches = glob.glob("/kaggle/input/**/Hanoi_Clean_Tabular.csv", recursive=True)
    if matches:
        return os.path.dirname(matches[0])

    # 2. Tìm kiếm theo vị trí file split_data.py trên máy local
    script_dir = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else "."
    candidates = [
        os.path.join(script_dir, "output"),
        os.path.join(script_dir, "dataset_kaggle"),
        script_dir,
        "./output",
        ".",
        "/Users/mac/Downloads/Thai/KLTN/output"
    ]
    for c_dir in candidates:
        if os.path.exists(os.path.join(c_dir, "Hanoi_Clean_Tabular.csv")):
            return c_dir

    # 3. Thông báo lỗi chi tiết nếu chưa gắn Input trên Kaggle
    if os.path.exists("/kaggle/input"):
        datasets = os.listdir("/kaggle/input")
        if not datasets:
            raise FileNotFoundError(
                "\n" + "=" * 70 + "\n"
                "⚠️ LỖI: Chưa có Dataset nào được gắn vào Notebook trên Kaggle!\n"
                "👉 Hướng dẫn khắc phục:\n"
                "   1. Nhìn sang thanh menu bên phải của Kaggle (cột Notebook).\n"
                "   2. Tại mục 'Input' -> Bấm nút '+ Add Input'.\n"
                "   3. Chọn Dataset 'Your Datasets' mà bạn đã upload.\n"
                + "=" * 70
            )
        else:
            raise FileNotFoundError(
                f"Không tìm thấy file 'Hanoi_Clean_Tabular.csv' trong /kaggle/input/{datasets}. "
                "Vui lòng kiểm tra lại cấu trúc file nén khi upload!"
            )
    else:
        raise FileNotFoundError("Không tìm thấy thư mục chứa 3 file dữ liệu sạch đầu vào!")


def main():
    print("=" * 70)
    print("BẮT ĐẦU QUY TRÌNH PHÂN CHIA DỮ LIỆU ĐỒNG BỘ (3 FILE VÀO -> 9 FILE RA)")
    print("=" * 70)

    # 1. XÁC ĐỊNH THƯ MỤC INPUT VÀ OUTPUT TỰ ĐỘNG
    input_dir = find_input_directory()
    output_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "./dataset_kaggle"
    os.makedirs(output_dir, exist_ok=True)

    print(f"Thư mục Input Dataset : {input_dir}")
    print(f"Thư mục Output (9 file): {output_dir}\n")

    # 2. ĐỌC 3 FILE ĐẦU VÀO
    tab_path  = os.path.join(input_dir, "Hanoi_Clean_Tabular.csv")
    text_path = os.path.join(input_dir, "Hanoi_Clean_Text.csv")
    full_path = os.path.join(input_dir, "Hanoi_Clean_Full.csv")

    for p in [tab_path, text_path, full_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Không tìm thấy file: {p}")

    tab_rows, tab_fields   = load_csv(tab_path)
    text_rows, text_fields = load_csv(text_path)
    full_rows, full_fields = load_csv(full_path)

    # DYNAMIC: Độ dài lấy trực tiếp từ dữ liệu thực tế, không hardcode
    n_total = len(tab_rows)
    print(f"Đã nạp thành công dữ liệu động (Tổng số bản ghi: {n_total:,} dòng):")
    print(f"  [1] Tabular : {len(tab_rows):,} dòng x {len(tab_fields)} cột")
    print(f"  [2] Text    : {len(text_rows):,} dòng x {len(text_fields)} cột")
    print(f"  [3] Full    : {len(full_rows):,} dòng x {len(full_fields)} cột")

    # 3. XÁC ĐỊNH SPLIT_MAP DỰA TRÊN TABULAR (NEAR-DUP VÀ PHÂN TẦNG ĐA CHIỀU)
    random.seed(42)

    near_dup_ids = set(r["id"] for r in tab_rows if r.get("is_near_dup", "").strip().lower() == "true")
    unique_rows  = [r for r in tab_rows if r["id"] not in near_dup_ids]

    # Gán near-duplicate vào Train để cách ly rò rỉ đề thi
    split_map = {idx: "train" for idx in near_dup_ids}

    # Phân tầng theo property_type x geo_group
    strata_map = defaultdict(list)
    for r in unique_rows:
        s = f"{r.get('property_type', 'unknown')}__{r.get('geo_group', 'unknown')}"
        strata_map[s].append(r["id"])

    strata_summary = []
    for strata, ids in sorted(strata_map.items()):
        random.shuffle(ids)
        n = len(ids)
        n_test = int(round(n * 0.10))
        n_val  = int(round(n * 0.10))

        test_ids  = ids[:n_test]
        val_ids   = ids[n_test : n_test + n_val]
        train_ids = ids[n_test + n_val :]

        for i in test_ids:  split_map[i] = "test"
        for i in val_ids:   split_map[i] = "val"
        for i in train_ids: split_map[i] = "train"

        strata_summary.append({
            "strata": strata, "total": n,
            "train": len(train_ids), "val": len(val_ids), "test": len(test_ids)
        })

    counts = Counter(split_map.values())
    print("\n" + "=" * 70)
    print(f"KẾT QUẢ ĐỒNG BỘ SPLIT TRÊN TỔNG SỐ {n_total:,} BẢN GHI THỰC TẾ:")
    print(f"  * TRAIN : {counts['train']:,} dòng ({counts['train']/n_total*100:.2f}%) [Gồm unique + {len(near_dup_ids):,} near-duplicates]")
    print(f"  * VAL   : {counts['val']:,} dòng ({counts['val']/n_total*100:.2f}%) [100% Unique]")
    print(f"  * TEST  : {counts['test']:,} dòng ({counts['test']/n_total*100:.2f}%) [100% Unique]")
    print("=" * 70)

    # In chi tiết phân bổ strata
    print("\nChi tiết phân tầng 12 Strata (property_type x geo_group):")
    print(f"{'Tổ hợp Strata':32s} | {'Tổng':6s} | {'Train':6s} | {'Val':5s} | {'Test':5s}")
    print("-" * 65)
    for s in strata_summary:
        print(f"{s['strata']:32s} | {s['total']:6d} | {s['train']:6d} | {s['val']:5d} | {s['test']:5d}")
    print("-" * 65)

    # 4. TẠO VÀ XUẤT ĐÚNG 9 FILE ĐỘC LẬP
    print("\nĐang xuất đúng 9 file đầu ra độc lập...")

    def split_and_save(prefix, rows, fields):
        for split_name in ["train", "val", "test"]:
            sub_rows = [r for r in rows if split_map.get(r["id"]) == split_name]
            fname = f"Hanoi_{prefix}_{split_name.capitalize()}.csv"
            fpath = os.path.join(output_dir, fname)
            write_csv(fpath, sub_rows, fields)
            print(f"  -> {fname:26s} : {len(sub_rows):,d} dòng x {len(fields)} cột")

    print("\n--- [NHÓM 1] TABULAR (3 files) ---")
    split_and_save("Tabular", tab_rows, tab_fields)

    print("\n--- [NHÓM 2] TEXT (3 files) ---")
    split_and_save("Text", text_rows, text_fields)

    print("\n--- [NHÓM 3] FULL MULTIMODAL (3 files) ---")
    split_and_save("Full", full_rows, full_fields)

    print("\n" + "=" * 70)
    print("HOÀN TẤT XUẤT 9 FILE DỮ LIỆU ĐỒNG BỘ 100%!")
    print(f"Toàn bộ file đã được lưu tại: {output_dir}")
    print("=" * 70)


if __name__ == "__main__":
    main()

BẮT ĐẦU QUY TRÌNH PHÂN CHIA DỮ LIỆU ĐỒNG BỘ (3 FILE VÀO -> 9 FILE RA)
Thư mục Input Dataset : /kaggle/input/datasets/hanhatthai/hanoidataset
Thư mục Output (9 file): /kaggle/working

Đã nạp thành công dữ liệu động (Tổng số bản ghi: 27,947 dòng):
  [1] Tabular : 27,947 dòng x 25 cột
  [2] Text    : 27,947 dòng x 5 cột
  [3] Full    : 27,947 dòng x 29 cột

KẾT QUẢ ĐỒNG BỘ SPLIT TRÊN TỔNG SỐ 27,947 BẢN GHI THỰC TẾ:
  * TRAIN : 22,543 dòng (80.66%) [Gồm unique + 940 near-duplicates]
  * VAL   : 2,702 dòng (9.67%) [100% Unique]
  * TEST  : 2,702 dòng (9.67%) [100% Unique]

Chi tiết phân tầng 12 Strata (property_type x geo_group):
Tổ hợp Strata                    | Tổng   | Train  | Val   | Test 
-----------------------------------------------------------------
condo__central                   |    931 |    745 |    93 |    93
condo__near_central              |   5003 |   4003 |   500 |   500
condo__new_urban                 |   4576 |   3660 |   458 |   458
condo__suburban                  